In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import os

# 1. Base de datos global
if 'respuestas_mercado' not in globals():
    respuestas_mercado = []

# 2. Opciones de mercado (Gama Alta 2025)
marcas_2025 = [
    'Selecciona una opción...',
    'Apple (iPhone 17 Pro/Max)',
    'Samsung (Galaxy S25 Ultra)',
    'Huawei (Mate 80 Pro / Pura 80)',
    'Xiaomi (Xiaomi 15 Ultra)',
    'Google (Pixel 10 Pro)'
]

# 3. Creación de Widgets (Interfaz)
titulo = widgets.HTML(
    value="<h2 style='color: #2c3e50;'>Estudio de Percepción: Mejor Smartphone Gama Alta 2025</h2>"
          "<p>¿Cuál consideras que es la mejor marca en la gama premium este año?</p><hr>"
)

# Controles de votación
dropdown_marca = widgets.Dropdown(options=marcas_2025, value='Selecciona una opción...', description='Marca:')
slider_edad = widgets.IntSlider(value=25, min=15, max=90, step=1, description='Tu Edad:')
btn_votar = widgets.Button(description='Registrar Voto', button_style='success', icon='check')

# Botones de Administración
btn_mostrar = widgets.Button(description='Mostrar top', button_style='info', icon='list')
btn_exportar = widgets.Button(description='Exportar datos', button_style='primary', icon='download')
btn_limpiar = widgets.Button(description='Limpiar datos', button_style='warning', icon='trash')
btn_salir = widgets.Button(description='Salir', button_style='danger', icon='times')

# Áreas de salida (pantallas dinámicas)
area_mensajes = widgets.Output() # Para confirmar votos o acciones
area_tabla = widgets.Output()    # Para mostrar el top cuando se solicite
interfaz_principal = widgets.Output() # Contenedor maestro para poder cerrar la app

# 4. Funciones lógicas de los botones
def votar(b):
    with area_mensajes:
        clear_output()
        marca = dropdown_marca.value
        if marca == 'Selecciona una opción...':
            print("⚠️ Selecciona una marca válida.")
            return

        respuestas_mercado.append({'Edad': slider_edad.value, 'Marca': marca})
        print("🎉 ¡Voto registrado exitosamente! Gracias por participar.")

        # Limpiamos el área de la tabla para que no quede visible automáticamente
        with area_tabla:
            clear_output()

def mostrar_top(b):
    with area_tabla:
        clear_output()
        if not respuestas_mercado:
            print("📊 Aún no hay datos para mostrar.")
        else:
            df = pd.DataFrame(respuestas_mercado)

            # Agrupamos por Marca y calculamos estadísticas de Edad
            top_marcas = df.groupby('Marca').agg(
                Votos=('Marca', 'count'),
                Edad_Promedio=('Edad', 'mean'),
                Edad_Minima=('Edad', 'min'),
                Edad_Maxima=('Edad', 'max')
            ).reset_index()

            # Redondeamos la edad promedio para que se vea mejor
            top_marcas['Edad_Promedio'] = top_marcas['Edad_Promedio'].round(1)

            # Ordenamos de mayor a menor por votos
            top_marcas = top_marcas.sort_values(by='Votos', ascending=False)

            display(widgets.HTML("<h3>🏆 Top de Preferencias Actuales</h3>"))
            display(top_marcas)

def exportar_datos(b):
    with area_mensajes:
        clear_output()
        if not respuestas_mercado:
            print("⚠️ No hay datos para exportar.")
            return
        df = pd.DataFrame(respuestas_mercado)
        nombre_archivo = 'datos_mercado_2025.csv'
        df.to_csv(nombre_archivo, index=False)
        print(f"✅ Datos exportados como '{nombre_archivo}'.")
        print("👉 Ve al ícono de la carpeta en el menú izquierdo de Colab para descargar tu archivo.")

def limpiar_datos(b):
    global respuestas_mercado
    respuestas_mercado.clear()
    with area_mensajes:
        clear_output()
        print("🗑️ Los datos de prueba han sido eliminados. La base está en 0.")
    with area_tabla:
        clear_output()

def salir_programa(b):
    with interfaz_principal:
        clear_output()
        display(widgets.HTML("<h3 style='color: #c0392b;'>🛑 Programa finalizado. Interfaz cerrada.</h3>"))

# 5. Conectar botones con sus funciones
btn_votar.on_click(votar)
btn_mostrar.on_click(mostrar_top)
btn_exportar.on_click(exportar_datos)
btn_limpiar.on_click(limpiar_datos)
btn_salir.on_click(salir_programa)

# 6. Organizar el diseño (Layout) y mostrar
fila_votacion = widgets.HBox([slider_edad, dropdown_marca, btn_votar])
fila_admin = widgets.HBox([btn_mostrar, btn_exportar, btn_limpiar, btn_salir])

with interfaz_principal:
    display(titulo)
    display(fila_votacion)
    display(area_mensajes)
    display(widgets.HTML("<hr><b>Panel de Administración (Solo Analistas)</b>"))
    display(fila_admin)
    display(area_tabla)

# Iniciar la app
display(interfaz_principal)

Output()